In [1]:
import pandas as pd
from biocypher import BioCypher

INFO -- This is BioCypher v0.9.6.
INFO -- Logging into `biocypher-log/biocypher-20250610-181414.log`.


In [2]:
print("Loading data from kg_giant.csv...")
file_path = 'data/kg_giant.csv'

try:
    df = pd.read_csv(file_path, dtype=str)
    print("Data loaded successfully. Here are the first 5 rows:")
    #display(df.head()) # 'display()' is a Jupyter function for nice tables
    print(df.head())
    print("\nAnd here is the data summary:")
    df.info()
except FileNotFoundError as e:
    print(f"Error: {e}. Make sure 'kg_giant.csv' is in the 'data' directory.")

Loading data from kg_giant.csv...
Data loaded successfully. Here are the first 5 rows:
          relation display_relation  x_id        x_type  x_name x_source  \
0  protein_protein              ppi  9796  gene/protein  PHYHIP     NCBI   
1  protein_protein              ppi  7918  gene/protein  GPANK1     NCBI   
2  protein_protein              ppi  8233  gene/protein   ZRSR2     NCBI   
3  protein_protein              ppi  4899  gene/protein    NRF1     NCBI   
4  protein_protein              ppi  5297  gene/protein   PI4KA     NCBI   

    y_id        y_type  y_name y_source  
0  56992  gene/protein   KIF15     NCBI  
1   9240  gene/protein   PNMA1     NCBI  
2  23548  gene/protein   TTC33     NCBI  
3  11253  gene/protein  MAN1B1     NCBI  
4   8601  gene/protein   RGS20     NCBI  

And here is the data summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8196862 entries, 0 to 8196861
Data columns (total 10 columns):
 #   Column            Dtype 
---  ------            ----- 


In [3]:
# In the next cell of your Jupyter Notebook

print("Diagnosing unique types from the DataFrame...")

# Find unique node types from both source (x) and target (y) columns
source_types = df['x_type'].unique()
target_types = df['y_type'].unique()
all_node_types = sorted(list(set(source_types) | set(target_types))) # Combine and sort

# Find unique relationship types
all_relation_types = sorted(list(df['relation'].unique()))

print("\n--- DIAGNOSTIC REPORT ---")
print("\nFound the following UNIQUE NODE TYPES:")
for node_type in all_node_types:
    print(f"- {node_type}")

print("\n\nFound the following UNIQUE RELATIONSHIP TYPES:")
for rel_type in all_relation_types:
    print(f"- {rel_type}")

print("\n--- END OF REPORT ---")
print("\nPlease copy and paste this entire report (from '--- DIAGNOSTIC REPORT ---' to '--- END OF REPORT ---') in your response.")

Diagnosing unique types from the DataFrame...

--- DIAGNOSTIC REPORT ---

Found the following UNIQUE NODE TYPES:
- anatomy
- biological_process
- cellular_component
- disease
- drug
- effect/phenotype
- exposure
- gene/protein
- molecular_function
- pathway


Found the following UNIQUE RELATIONSHIP TYPES:
- anatomy_anatomy
- anatomy_protein_absent
- anatomy_protein_present
- bioprocess_bioprocess
- bioprocess_protein
- cellcomp_cellcomp
- cellcomp_protein
- contraindication
- disease_disease
- disease_phenotype_negative
- disease_phenotype_positive
- disease_protein
- drug_drug
- drug_effect
- drug_protein
- exposure_bioprocess
- exposure_cellcomp
- exposure_disease
- exposure_exposure
- exposure_molfunc
- exposure_protein
- indication
- molfunc_molfunc
- molfunc_protein
- off-label use
- pathway_pathway
- pathway_protein
- phenotype_phenotype
- phenotype_protein
- protein_protein

--- END OF REPORT ---

Please copy and paste this entire report (from '--- DIAGNOSTIC REPORT ---' to '---

#### Excuting actual conversion

In [4]:
print("Setting up BioCypher and data generators...")

# Instantiate BioCypher
bc = BioCypher(schema_config_path='config/schema_config.yaml')

# Node Generator (no changes needed here)
def generate_nodes():
    print("Preparing unique nodes...")
    source_nodes = df[['x_id', 'x_type', 'x_name', 'x_source']].rename(columns={
        'x_id': 'id', 'x_type': 'type', 'x_name': 'name', 'x_source': 'source'
    })
    target_nodes = df[['y_id', 'y_type', 'y_name', 'y_source']].rename(columns={
        'y_id': 'id', 'y_type': 'type', 'y_name': 'name', 'y_source': 'source'
    })
    unique_nodes_df = pd.concat([source_nodes, target_nodes]).drop_duplicates(subset=['id'])

    print("Yielding nodes...")
    for _, row in unique_nodes_df.iterrows():
        node_id = row['id']
        node_label = row['type'].replace('gene/protein', 'protein')
        properties = {'name': row['name'], 'source': row['source']}
        yield (node_id, node_label, properties)

# Edge Generator (CORRECTED)
def generate_edges():
    print("Yielding edges...")
    for _, row in df.iterrows():
        source_id = row['x_id']
        target_id = row['y_id']
        # FIX: Pass the raw relation label. BioCypher will handle mapping.
        edge_label = row['relation']
        properties = {'display_relation': row['display_relation']}
        yield (None, source_id, target_id, edge_label, properties)

print("Setup complete.")

INFO -- Running BioCypher with schema configuration from config/schema_config.yaml.


Setting up BioCypher and data generators...
Setup complete.


In [5]:
# Cell 4: Run the Import Process
print("Running the BioCypher import process...")
bc.write_nodes(generate_nodes())
bc.write_edges(generate_edges())
bc.write_import_call()
bc.summary()
print("\nScript finished! Check the 'output' directory for the generated files.")

INFO -- Loading ontologies...
INFO -- Instantiating OntologyAdapter class for https://github.com/biolink/biolink-model/raw/v3.2.1/biolink-model.owl.ttl.


Running the BioCypher import process...
Preparing unique nodes...


INFO -- Creating output directory `/home/sheraz/aieop/primekg-biocypher-importer/biocypher-out/20250610181443`.


Yielding nodes...


INFO -- Writing 26974 entries to Protein-part000.csv
INFO -- Writing 7957 entries to Drug-part000.csv
INFO -- Writing 6467 entries to Effect_phenotype-part000.csv
INFO -- Writing 11757 entries to Disease-part000.csv
INFO -- Writing 19759 entries to Biological_process-part000.csv
INFO -- Writing 6516 entries to Molecular_function-part000.csv
INFO -- Writing 2740 entries to Cellular_component-part000.csv
INFO -- Writing 818 entries to Exposure-part000.csv
INFO -- Writing 2516 entries to Pathway-part000.csv
INFO -- Writing 4850 entries to Anatomy-part000.csv


Yielding edges...


WARNING -- Duplicate edge type drug_protein found. 
WARNING -- Duplicate edge type disease_phenotype_positive found. 
WARNING -- Duplicate edge type disease_protein found. 
WARNING -- Duplicate edge type molfunc_protein found. 
WARNING -- Duplicate edge type cellcomp_protein found. 
WARNING -- Duplicate edge type bioprocess_protein found. 
WARNING -- Duplicate edge type anatomy_protein_present found. 
INFO -- Writing 1000000 entries to Drug_drug-part000.csv
INFO -- Writing 1000000 entries to Drug_drug-part001.csv
INFO -- Writing 1000000 entries to Anatomy_protein_present-part000.csv
INFO -- Writing 1000000 entries to Anatomy_protein_present-part001.csv
INFO -- Writing 1000000 entries to Anatomy_protein_present-part002.csv
INFO -- Writing 642150 entries to Protein_protein-part000.csv
INFO -- Writing 50936 entries to Drug_protein-part000.csv
INFO -- Writing 67968 entries to Contraindication-part000.csv
INFO -- Writing 20122 entries to Indication-part000.csv
INFO -- Writing 5518 entries t

KeyboardInterrupt: 